# ZetaGo -- Kaggle training notebook 

Runs the Track A supervised sweep (`training/supervised/trainer.py`) on Kaggle instead of
Colab -- see `Docs/execution/EXECUTION_Phase2.md` for the full design. This notebook **replaces**
`notebooks/trainer.ipynb` (Colab, discarded); `notebooks/dataset_generator.ipynb` is unaffected
and stays on Colab (generation is a finished, CPU/Eigen-only job, unrelated to where training
runs).

**Hard-won lesson baked into this version (31 July 2026):** an interactive Kaggle session's
`/kaggle/working` is **not** durable. Neither "revert to a saved version" nor "Quick Save" reliably
restores/persists it (both confirmed empty after a real interruption -- see
`Docs/execution/EXECUTION_Phase2.md` task 2.0i). The only mechanism that survives a full container wipe is a
real attached Kaggle **Dataset** -- so this notebook now pushes one **automatically, every few
minutes, while training runs**, instead of depending on a human downloading files at the right
moment.

**Before running:** attach two Kaggle Dataset inputs to this notebook (Add Data, right sidebar):

1. `zetago-code` -- the repo's training/eval code, pushed with `kaggle/sync_code.sh`. **Re-push
   and re-attach the latest version after any local code change** -- there is no automatic sync.
2. `zetago-dataset-7x7` -- the frozen `train/val/test.h5` + `DATASET_CARD.md`, pushed once with
   `kaggle/sync_dataset.sh`.

Do **not** attach `zetago-checkpoints` on your very first run of this redesigned notebook -- it
currently only holds placeholder test data from verifying the push mechanism, not real progress.
Once Step 5 runs for real and pushes at least once, it's safe to attach on every session after
that; Step 3 picks it up automatically.

**One-time setup, before Step 2 will work:** add a Kaggle Secret (Add-ons -> Secrets, top menu)
named `KAGGLE_KEY` containing your Kaggle API token (the same value you already use locally at
`~/.kaggle/access_token`). This lets Step 2 authenticate to Kaggle's own API from inside the
notebook without the token ever appearing in this notebook's saved source. Do this once; it stays
attached to this notebook for every future session.

## Internet ON vs Internet OFF, and CPU vs GPU

- **Internet must stay ON for this entire notebook now**, not just Step 1 -- Step 2's credential
  check and Step 5's auto-checkpoint push both need to reach Kaggle's own API. This is a
  deliberate change from the original design (which kept Internet OFF during training to prove no
  external dependency): after losing real progress to session fragility, durability now takes
  priority over that reproducibility nicety. Nothing else needs internet -- still no `git clone`,
  no `pip install` in the normal path (Step 2 will self-install the `kaggle` CLI via pip if it's
  ever missing from the base image, since internet is on anyway), no external service other than
  Kaggle's own API.
- **CPU vs GPU** is a Settings -> Accelerator choice (None / GPU T4 x2). `--device auto` in
  `trainer.py` already picks up whichever is available; Step 5 has both a single-accelerator cell
  and a two-T4 background-worker variant.

In [ ]:
import os
import sys

# No git clone, no pip install by default -- code comes from the attached
# Dataset input. Found the hard way (a live run 403'd/FileNotFound'd on the
# "classic" documented path): this Kaggle workspace mounts private datasets
# at /kaggle/input/datasets/<owner>/<slug>, not /kaggle/input/<slug>. Check
# both, plus a fallback scan, so this notebook doesn't silently break again
# if the mount convention differs on a different Kaggle workspace/account.
def _find_mount(slug):
    candidates = [os.path.join("/kaggle/input", slug)]
    datasets_root = "/kaggle/input/datasets"
    if os.path.isdir(datasets_root):
        for owner in os.listdir(datasets_root):
            candidates.append(os.path.join(datasets_root, owner, slug))
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(
        f"Kaggle input dataset '{slug}' not found. Checked: {candidates}. "
        "Is it attached under Input on the right sidebar? Try restarting the session "
        "if you just attached it."
    )

CODE_DIR = _find_mount("zetago-code")
DATA_DIR = _find_mount("zetago-dataset-7x7")

sys.path.insert(0, CODE_DIR)
print("CODE_DIR:", CODE_DIR, "->", os.listdir(CODE_DIR))
print("DATA_DIR:", DATA_DIR, "->", os.listdir(DATA_DIR))

## Step 1 -- environment check (run once per Kaggle base image, not per session)

Confirms the versions Kaggle's default image ships match what `requirements.txt` expects, and
whether a GPU is actually visible.

In [ ]:
import h5py
import numpy as np
import sklearn
import torch

print("numpy:", np.__version__)
print("h5py:", h5py.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i))

# Only expected to be needed if a version check above looks wrong -- not part
# of the normal path.
# !pip install -q -r {CODE_DIR}/requirements.txt

## Step 2 -- Kaggle credentials for auto-checkpointing

Reads the `KAGGLE_KEY` secret (Add-ons -> Secrets -- see the intro cell if you haven't added it
yet) and writes it to `~/.kaggle/access_token`, exactly like the local setup that already makes
`kaggle/sync_code.sh`/`sync_dataset.sh` work. Installs the `kaggle` CLI via pip if this base image
doesn't already have it (internet is on anyway). Verifies it authenticates before anything depends
on it -- a bad or missing secret should fail loudly here, not silently 10 minutes into Step 5.

In [ ]:
import os
import shutil
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    kaggle_key = user_secrets.get_secret("KAGGLE_KEY")
except Exception as exc:
    raise RuntimeError(
        "Couldn't read the 'KAGGLE_KEY' secret. Add it via Add-ons -> Secrets (top menu): "
        "name it exactly KAGGLE_KEY, value = your Kaggle API token, then attach it to this "
        "notebook and re-run this cell."
    ) from exc

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
token_path = os.path.expanduser("~/.kaggle/access_token")
with open(token_path, "w") as f:
    f.write(kaggle_key)
os.chmod(token_path, 0o600)

if shutil.which("kaggle") is None:
    print("kaggle CLI not found in this image -- installing (internet is required to be ON)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

check = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError(f"Kaggle credentials didn't authenticate:\n{check.stderr}")
print("Kaggle credentials OK -- auto-checkpointing (Step 5) can push to your account.")

## Step 3 -- resume setup

If a `zetago-checkpoints` dataset from a previous session on this sweep is attached, copy its
`results/` and `checkpoints/` into `/kaggle/working/` so `--resume`/`--checkpoint-dir` (Step 5)
pick up where the last session left off. Safe to run even with nothing attached -- it just leaves
`/kaggle/working` empty and Step 5 starts fresh (expected and fine on this notebook's very first
run, before Step 5 has ever pushed a `zetago-checkpoints` dataset).

In [ ]:
import os
import shutil

os.makedirs("/kaggle/working/results", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

prev = None
for base in ("/kaggle/input/zetago-checkpoints", "/kaggle/input/datasets"):
    if base == "/kaggle/input/datasets" and os.path.isdir(base):
        for owner in os.listdir(base):
            candidate = os.path.join(base, owner, "zetago-checkpoints")
            if os.path.isdir(candidate):
                prev = candidate
                break
    elif os.path.isdir(base):
        prev = base

# push_checkpoint() (Step 5) stages files flat with a "results__"/"checkpoints__" prefix, since
# Kaggle doesn't reliably preserve subfolders on upload -- reconstruct the two folders from that
# prefix. Fall back to sorting by extension for datasets pushed by an older, unprefixed
# push_checkpoint() (a live kernel started before this fix keeps running its original in-memory
# version regardless of what the notebook file now says).
restored = {"results": [], "checkpoints": []}
if prev:
    for fn in os.listdir(prev):
        matched = False
        for name in ("results", "checkpoints"):
            prefix = f"{name}__"
            if fn.startswith(prefix):
                dest_name = fn[len(prefix):]
                shutil.copy(os.path.join(prev, fn), f"/kaggle/working/{name}/{dest_name}")
                restored[name].append(dest_name)
                matched = True
        if not matched:
            dest = "checkpoints" if fn.endswith(".pt") else "results"
            shutil.copy(os.path.join(prev, fn), f"/kaggle/working/{dest}/{fn}")
            restored[dest].append(fn)
    print(f"Resumed from a previous zetago-checkpoints session ({prev}):")
    print(" results:", restored["results"])
    print(" checkpoints:", restored["checkpoints"])
else:
    print("No zetago-checkpoints input attached -- starting fresh (expected on the first-ever run).")

## Step 4 -- device check

Confirms which accelerator this session actually has before committing to the single-command or
two-T4 variant below.

In [ ]:
import torch

if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print("2+ GPUs visible -- use the two-T4 cell in Step 5.")
elif torch.cuda.is_available():
    print("1 GPU visible -- use the single-accelerator cell in Step 5 with --device cuda.")
else:
    print("No GPU visible -- use the single-accelerator cell in Step 5 with --device cpu "
          "(or --device auto, which resolves the same way).")

## Step 5 -- run the sweep, with automatic checkpointing

Both cells pass `--resume` and `--checkpoint-dir` (`Docs/execution/EXECUTION_Phase2.md` tasks 2.0d/2.0e), and
both run training as a **background process** so this same cell can also run a loop that pushes
`/kaggle/working/{results,checkpoints}` to the real `zetago-checkpoints` Dataset every
`CHECKPOINT_INTERVAL_SECONDS` (default 10 minutes) for as long as training runs, plus once more
right after it finishes. This is now the actual safety net -- not Quick Save, not version-revert,
not a manual download. Worst case if the container dies without warning: the last ~10 minutes of
progress, not the whole session.

First the shared push helper, then the training cell:

In [ ]:
import json
import os
import shutil
import subprocess
import time

CHECKPOINT_STAGING = "/kaggle/working/_checkpoint_staging"
CHECKPOINT_DATASET_ID = "obiditislam/zetago-checkpoints"
CHECKPOINT_INTERVAL_SECONDS = 600  # 10 minutes -- not tuned, just a sane default; lower it if a
                                    # single session's compute budget makes 10 minutes of redone
                                    # work feel expensive, higher it if pushes start feeling chatty.
POLL_SECONDS = 30  # how often to check whether training finished, independent of push cadence

def push_checkpoint():
    os.makedirs(CHECKPOINT_STAGING, exist_ok=True)
    for fn in os.listdir(CHECKPOINT_STAGING):
        if fn != "dataset-metadata.json":
            os.remove(os.path.join(CHECKPOINT_STAGING, fn))
    with open(f"{CHECKPOINT_STAGING}/dataset-metadata.json", "w") as f:
        json.dump(
            {"title": "ZetaGo checkpoints", "id": CHECKPOINT_DATASET_ID, "licenses": [{"name": "CC0-1.0"}]},
            f,
        )

    # Kaggle does not reliably preserve subfolder structure on upload (confirmed: results/*.json
    # pushed via `kaggle datasets version` landed at the dataset root, not under results/, per
    # `kaggle datasets files`). So stage everything flat ourselves with a name__ prefix recording
    # which /kaggle/working subfolder it came from, and let Step 3 reconstruct the folders on
    # restore -- don't depend on Kaggle preserving nesting at all.
    n_staged = 0
    for name in ("results", "checkpoints"):
        src = f"/kaggle/working/{name}"
        if os.path.isdir(src):
            for fn in os.listdir(src):
                shutil.copy(os.path.join(src, fn), os.path.join(CHECKPOINT_STAGING, f"{name}__{fn}"))
                n_staged += 1

    msg = "auto " + time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    r = subprocess.run(
        ["kaggle", "datasets", "version", "-p", CHECKPOINT_STAGING, "-m", msg, "-r", "zip", "-t"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        subprocess.run(
            ["kaggle", "datasets", "create", "-p", CHECKPOINT_STAGING, "-r", "zip", "-t"],
            capture_output=True, text=True,
        )

    n_rows = "?"
    metrics_path = "/kaggle/working/results/supervised_track_a_metrics.json"
    if os.path.exists(metrics_path):
        try:
            with open(metrics_path) as f:
                n_rows = len(json.load(f))
        except Exception:
            pass
    print(f"[auto-checkpoint] pushed {n_staged} file(s) at {msg} ({n_rows} rows in results)")

def run_with_auto_checkpoint(train_procs):
    last_push = time.monotonic()
    while any(p.poll() is None for p in train_procs):
        time.sleep(POLL_SECONDS)
        if time.monotonic() - last_push >= CHECKPOINT_INTERVAL_SECONDS:
            try:
                push_checkpoint()
            except Exception as e:
                print("[auto-checkpoint] push failed, will retry next interval:", e)
            last_push = time.monotonic()
    push_checkpoint()
    for i, p in enumerate(train_procs):
        print(f"process {i} exit code:", p.returncode)

**Run the sweep** (single accelerator -- CPU or one GPU):

In [ ]:
import subprocess
import sys

train_cmd = [
    sys.executable, f"{CODE_DIR}/training/train_supervised.py",
    "--train-h5", f"{DATA_DIR}/train.h5",
    "--val-h5", f"{DATA_DIR}/val.h5",
    "--model", "all", "--encodings", "2", "4", "7",
    "--volumes", "1000", "5000", "20000", "full",
    "--seeds", "42", "43", "44", "45", "46",
    "--dedup", "none",
    "--resume", "--checkpoint-dir", "/kaggle/working/checkpoints",
    "--out-csv", "/kaggle/working/results/supervised_track_a_metrics.csv",
    "--out-json", "/kaggle/working/results/supervised_track_a_metrics.json",
    "--device", "auto",
]
train_proc = subprocess.Popen(train_cmd)
run_with_auto_checkpoint([train_proc])

## Step 6 -- when the sweep is done (or you're stopping for the day)

Auto-checkpointing means there's no longer a manual "did I save recently enough" step -- the
`zetago-checkpoints` Dataset is already at most `CHECKPOINT_INTERVAL_SECONDS` behind whatever's
actually happened. A few things still worth doing:

1. **Save Version** (top right) is still fine to do for your own convenience (keeps this
   notebook's code/cell-output history readable) -- just don't rely on it for data, and don't use
   "Save & Run All (Commit)" while Step 5 is still running (it starts a fresh session and abandons
   the running training).
2. Once `--out-json` shows all 60 CNN cells (+ classical baselines) present, download
   `results/supervised_track_a_metrics.{csv,json}` for task 2.3's consolidation and task 2.6's
   sanity check -- or just pull the latest `zetago-checkpoints` Dataset version locally with
   `kaggle datasets download obiditislam/zetago-checkpoints`.
3. Starting a new session later: attach `zetago-checkpoints` (it now always exists once Step 5 has
   run at least once) alongside the other two datasets -- Step 3 restores it automatically.

## Step 7 -- persist the champion model

The 300-cell sweep (Step 5) recorded **metrics only** -- `trainer.py` deletes each cell's
`--checkpoint-dir` file on successful completion, so a finished sweep leaves no weights behind and
task 2.4 (MCTS round-robin) has nothing to wrap. This step re-runs the single best cell with
`--save-model-dir`, which persists final weights, and pushes them as their own Dataset.

Champion from the sweep: **`cnn`, N=4, seed=46, full volume** -- top-1 0.9046, value acc 0.9151,
the best single cell of 300. Costs ~13 min on the GPU. Factor A is null for the CNN (N=2/4/7 all
within seed noise), so this is "best observed" rather than "meaningfully better than N=2".

**Requires `zetago-code` v2 or later attached** -- `--save-model-dir` does not exist in v1.

In [ ]:
import os
import subprocess
import sys

os.makedirs("/kaggle/working/models", exist_ok=True)

champion_cmd = [
    sys.executable, f"{CODE_DIR}/training/train_supervised.py",
    "--train-h5", f"{DATA_DIR}/train.h5",
    "--val-h5", f"{DATA_DIR}/val.h5",
    "--model", "cnn", "--encodings", "4",
    "--volumes", "full", "--seeds", "46",
    "--dedup", "none",
    "--save-model-dir", "/kaggle/working/models",
    "--out-csv", "/kaggle/working/results/champion.csv",
    "--out-json", "/kaggle/working/results/champion.json",
    "--device", "auto",
]
print("Training the champion cell (~13 min on GPU)...")
subprocess.run(champion_cmd, check=True)
print("\nSaved model files:", os.listdir("/kaggle/working/models"))

In [ ]:
import json
import os
import subprocess

MODELS_DATASET_ID = "obiditislam/zetago-models"
MODELS_STAGING = "/kaggle/working/_models_staging"

os.makedirs(MODELS_STAGING, exist_ok=True)
with open(f"{MODELS_STAGING}/dataset-metadata.json", "w") as f:
    json.dump(
        {"title": "ZetaGo models", "id": MODELS_DATASET_ID, "licenses": [{"name": "CC0-1.0"}]},
        f,
    )

# Kept separate from zetago-checkpoints on purpose: that dataset is rewritten wholesale every 10
# minutes by Step 5's auto-checkpoint loop, so anything left there would be clobbered by the next
# sweep. Champion weights need to outlive any single run.
import shutil

n = 0
for fn in os.listdir("/kaggle/working/models"):
    shutil.copy(f"/kaggle/working/models/{fn}", f"{MODELS_STAGING}/{fn}")
    n += 1
if n == 0:
    raise RuntimeError("nothing in /kaggle/working/models -- run the training cell above first")

r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", MODELS_STAGING, "-m", "champion cnn N=4 seed=46 full", "-r", "zip", "-t"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    r = subprocess.run(
        ["kaggle", "datasets", "create", "-p", MODELS_STAGING, "-r", "zip", "-t"],
        capture_output=True, text=True,
    )
print(r.stdout or r.stderr)
print(f"pushed {n} model file(s) to {MODELS_DATASET_ID}")